In [1]:
import sqlite3

import pandas as pd
from bertopic import BERTopic

conn = sqlite3.connect("db/axiom.db")  # Creates or opens the database file
cursor = conn.cursor()

cursor.execute('''
    SELECT speech, summary
    FROM speeches s
    ORDER BY date
''')

data = cursor.fetchall()
conn.close()

speeches = []
summary = []
for entry in data:
    speeches.append(entry[0])
    summary.append(entry[1])

base = BERTopic()
base_topics, base_probs = base.fit_transform(summary[0:7000])

new = BERTopic()
new_topics, new_probs = new.fit_transform(summary[7000:8975])

merged_model = BERTopic.merge_models([base, new])

/home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
merged_model.update_topics(summary)
merged_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2333,-1_the_and_to_of,"[the, and, to, of, for, in, that, with, on, is]",NaN
1,0,348,0_singapore_trade_economic_smes,"[singapore, trade, economic, smes, china, asea...",NaN
2,1,214,1_questions_nos_permission_requested,"[questions, nos, permission, requested, combin...",NaN
3,2,164,2_mas_financial_moneylenders_bill,"[mas, financial, moneylenders, bill, financing...",NaN
4,3,132,3_saf_ns_defence_mindef,"[saf, ns, defence, mindef, servicemen, ng, tra...",NaN
...,...,...,...,...,...
142,141,11,141_flexi_flats_room_singles,"[flexi, flats, room, singles, elderly, applica...",NaN
143,142,11,142_women_gender_she_men,"[women, gender, she, men, female, diversity, e...",NaN
144,143,10,143_vehicles_lta_vehicle_recall,"[vehicles, lta, vehicle, recall, inspections, ...",NaN
145,144,33,144_nric_numbers_acra_statement,"[nric, numbers, acra, statement, ministerial, ...",NaN


In [3]:
ds = []
for entry in merged_model.get_topic_info().itertuples(index=True):
    ds.append({
        "topic": entry.Name,
        "representation": entry.Representation
    })

print(ds)

[{'topic': '-1_the_and_to_of', 'representation': ['the', 'and', 'to', 'of', 'for', 'in', 'that', 'with', 'on', 'is']}, {'topic': '0_singapore_trade_economic_smes', 'representation': ['singapore', 'trade', 'economic', 'smes', 'china', 'asean', 'companies', 'global', 'innovation', 'growth']}, {'topic': '1_questions_nos_permission_requested', 'representation': ['questions', 'nos', 'permission', 'requested', 'combine', 'speaker', 'question', 'together', 'discussion', 'address']}, {'topic': '2_mas_financial_moneylenders_bill', 'representation': ['mas', 'financial', 'moneylenders', 'bill', 'financing', 'money', 'laundering', 'regulatory', 'singapore', 'act']}, {'topic': '3_saf_ns_defence_mindef', 'representation': ['saf', 'ns', 'defence', 'mindef', 'servicemen', 'ng', 'training', 'nsmen', 'forces', 'armed']}, {'topic': '4_care_seniors_ageing_home', 'representation': ['care', 'seniors', 'ageing', 'home', 'nursing', 'moh', 'caregivers', 'dementia', 'services', 'health']}, {'topic': '5_patients

In [4]:
from distilabel.steps import (
    GroupColumns,
    KeepColumns,
    ExpandColumns,
    PushToHub,
    make_generator_step
)
from distilabel.pipeline import Pipeline
from custom_modules.CustomLLMs import OpenRouterLLM
from custom_modules.utils import ExtractPythonArray, TemplateFormatter, ToJsonFile
from templates.SFT_templates import TOPIC_LABEL_TEMPLATE

with Pipeline(name="generate_topic_labels") as topic_label_pipeline:
    fromds = make_generator_step(
        ds,
        output_mappings={
            "representation": "keywords"
        }
    )

    formatter = TemplateFormatter(
        template=TOPIC_LABEL_TEMPLATE,
        template_inputs=["keywords"]
    )

    llm = OpenRouterLLM(
        model="qwen/qwen-2.5-72b-instruct",
        max_tokens=1024,
        max_workers=50,
        temperature=0.0001
    )

    extract = ExtractPythonArray()

    expand = ExpandColumns(
        columns={
            "array": "topic_label"
        }
    ) 

    keep_columns = KeepColumns(
        columns=["topic", "topic_label", "keywords"],
    )

    tojson = ToJsonFile(
        filepath="outputs/topic_labelling",
        filename="topic_labelling",
        jsonl=False
    )

    fromds >> formatter >> llm >> extract >> expand >> keep_columns >> tojson

topic_label_pipeline.run(use_cache=False)

INFO 08-06 11:22:25 [__init__.py:244] Automatically detected platform cuda.


2025-08-06 11:22:27,495	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


[08/06/25 11:22:27] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=395962;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=421614;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/tytan216/.cache/distilabel/pipelines/generate_topic_labels/e8e98e5             
                             47f0752a2c753a930ab8960f85e7aae39/executions/f06dd6c07eb73817d904b4407369             
                             04d33499697e/data/steps_outputs'                                                      

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=567307;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=489360;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_dicts_0'                                                      
                                - 🔄 'template_formatter_0'                                                        
                              * Stage 1:                                                                           
                                - 🌐 'open_router_l_l_m_0'                                                         
                              * Stage 2:                                                                           
                                - 🔄 'extract_python_array_0'                                                      
                                - 🔄 'expand_columns_0'                                                            
                                - 🔄 'keep_columns_0'                                                              
                              * Stage 3:                                                                           
                                - 🌐 'to_json_file_0'                                                              

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

...

                    INFO     ['distilabel.step.to_json_file_0'] 📦 Processing batch 0 in        ]8;id=963971;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=853070;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#230\230]8;;\
                             'to_json_file_0' (replica ID: 0)                                                      

                    INFO     ['distilabel.step.to_json_file_0'] 📨 Step 'to_json_file_0'        ]8;id=26347;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=213751;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             sending batch 0 to output queue                                                       

                    INFO     ['distilabel.step.to_json_file_0'] 🏁 Finished running step        ]8;id=325497;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=708970;file:///home/tytan216/volume/tzeyoung/Policy_RLHF/venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             'to_json_file_0' (replica ID: 0)                                                      

Generating train split: 735 examples [00:00, 226411.09 examples/s]


Distiset({
    default: DatasetDict({
        train: Dataset({
            features: ['topic', 'topic_label', 'keywords'],
            num_rows: 735
        })
    })
})